# 🎬 iromusic Recommender System

This notebook implements a movie recommender system using the data fetched from the iromusicapp.ir API.

## 1. Data Loading

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
# Load data
DATA_PATH = Path("iromusic_client/output/2026/02/20/movie")
json_files = list(DATA_PATH.glob("*.json"))
latest_file = max(json_files, key=lambda p: p.stat().st_mtime)
print(f"Loading: {latest_file.name}")

with open(latest_file, 'r', encoding='utf-8') as f:
    movie_data = json.load(f)

df = pd.DataFrame(movie_data)
print(f"Dataset: {df.shape[0]} movies")

Loading: movie_movies_2026-02-20_21-21-19.json
Dataset: 51 movies


## 2. Data Preprocessing

In [3]:
# Create clean copy
df_clean = df.copy()

# Convert numeric columns
df_clean['imdbRating_numeric'] = pd.to_numeric(df['imdbRating'], errors='coerce').fillna(0)
df_clean['views_numeric'] = pd.to_numeric(df['views'], errors='coerce').fillna(0)
df_clean['likes_numeric'] = pd.to_numeric(df['likes'], errors='coerce').fillna(0)
df_clean['year_numeric'] = pd.to_numeric(df['year'], errors='coerce').fillna(2020)

# Fill missing genres
df_clean['genres'] = df_clean['genres'].apply(lambda x: x if isinstance(x, list) else [])
df_clean['englishTitle'] = df_clean['englishTitle'].fillna('Unknown')

# Extract genre list
def extract_genres(genres_list):
    if isinstance(genres_list, list):
        return [g.get('title', 'Unknown') for g in genres_list if isinstance(g, dict)]
    return []

df_clean['genre_list'] = df_clean['genres'].apply(extract_genres)

# Get unique genres
all_unique_genres = set()
for genres in df_clean['genre_list']:
    all_unique_genres.update(genres)
all_unique_genres = sorted(list(all_unique_genres))

print(f"Unique genres: {len(all_unique_genres)}")
print(f"Genres: {all_unique_genres[:10]}...")

Unique genres: 17
Genres: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History']...


## 3. Feature Engineering

In [4]:
# Create genre one-hot encoding
for genre in all_unique_genres:
    df_clean[f'genre_{genre}'] = df_clean['genre_list'].apply(lambda x: 1 if genre in x else 0)

genre_cols = [col for col in df_clean.columns if col.startswith('genre_')]
print(f"Created {len(genre_cols)} genre features")

Created 18 genre features


In [5]:
# Create popularity score
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_clean['views_norm'] = scaler.fit_transform(df_clean[['views_numeric']])
df_clean['likes_norm'] = scaler.fit_transform(df_clean[['likes_numeric']])
df_clean['popularity_score'] = 0.6 * df_clean['views_norm'] + 0.4 * df_clean['likes_norm']

# Scale numeric features
scaler = MinMaxScaler()
numeric_cols = ['imdbRating_numeric', 'year_numeric', 'popularity_score']
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

print("Features created successfully!")

Features created successfully!


## 4. Content-Based Filtering

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Build feature matrix
feature_cols = genre_cols + numeric_cols
feature_matrix = df_clean[feature_cols].values

print(f"Feature matrix: {feature_matrix.shape}")

# Compute similarity
similarity_matrix = cosine_similarity(feature_matrix)
print(f"Similarity matrix: {similarity_matrix.shape}")

Feature matrix: (51, 21)


ValueError: setting an array element with a sequence.

In [ ]:
class ContentBasedRecommender:
    """Content-Based Filtering using cosine similarity"""
    
    def __init__(self, df, similarity_matrix):
        self.df = df
        self.similarity_matrix = similarity_matrix
        self.id_to_idx = {id_: idx for idx, id_ in enumerate(df['id'])}
    
    def get_similar_movies(self, movie_id, n=5):
        if movie_id not in self.id_to_idx:
            return None
        idx = self.id_to_idx[movie_id]
        sim_scores = self.similarity_matrix[idx]
        top_indices = np.argsort(sim_scores)[::-1][1:n+1]
        
        results = self.df.iloc[top_indices][['id', 'englishTitle', 'year', 'genre_list', 'imdbRating_numeric']].copy()
        results['similarity_score'] = sim_scores[top_indices]
        return results

content_recommender = ContentBasedRecommender(df_clean, similarity_matrix)
print("Content-Based Recommender ready!")

In [ ]:
# Test recommendations
sample_movie = df_clean.iloc[0]
print(f"Movie: {sample_movie['englishTitle']} ({sample_movie['year']})")
print(f"Genres: {sample_movie['genre_list']}")

recs = content_recommender.get_similar_movies(sample_movie['id'], n=5)
print("\nSimilar movies:")
print(recs[['englishTitle', 'year', 'similarity_score']].to_string(index=False))

## 5. Popularity-Based Recommender

In [ ]:
class PopularityRecommender:
    """Simple popularity-based recommender"""
    
    def __init__(self, df):
        self.df = df
    
    def get_top_rated(self, n=10):
        return self.df.nlargest(n, 'imdbRating_numeric')[['id', 'englishTitle', 'year', 'imdbRating_numeric', 'views_numeric', 'likes_numeric']]
    
    def get_most_viewed(self, n=10):
        return self.df.nlargest(n, 'views_numeric')[['id', 'englishTitle', 'year', 'views_numeric']]
    
    def get_most_liked(self, n=10):
        return self.df.nlargest(n, 'likes_numeric')[['id', 'englishTitle', 'year', 'likes_numeric']]

popularity_recommender = PopularityRecommender(df_clean)
print("Popularity Recommender ready!")

In [ ]:
print("Top Rated Movies:")
print(popularity_recommender.get_top_rated(5).to_string(index=False))

print("\nMost Viewed:")
print(popularity_recommender.get_most_viewed(5).to_string(index=False))

## 6. Hybrid Recommender

In [ ]:
class HybridRecommender:
    """Combines content-based and popularity-based"""
    
    def __init__(self, df, similarity_matrix):
        self.df = df
        self.similarity_matrix = similarity_matrix
        self.id_to_idx = {id_: idx for idx, id_ in enumerate(df['id'])}
    
    def recommend(self, movie_id=None, liked_genres=None, n=10, content_weight=0.6, pop_weight=0.4):
        scores = np.zeros(len(self.df))
        
        if movie_id and movie_id in self.id_to_idx:
            idx = self.id_to_idx[movie_id]
            scores += content_weight * self.similarity_matrix[idx]
        
        if liked_genres:
            def genre_match(genres):
                return sum(1 for g in genres if g in liked_genres) / max(len(genres), 1)
            genre_scores = self.df['genre_list'].apply(genre_match).values
            scores += content_weight * 0.3 * genre_scores
        
        scores += pop_weight * self.df['popularity_score'].values
        
        top_indices = np.argsort(scores)[::-1][:n]
        results = self.df.iloc[top_indices][['id', 'englishTitle', 'year', 'genre_list']].copy()
        results['score'] = scores[top_indices]
        return results

hybrid_recommender = HybridRecommender(df_clean, similarity_matrix)
print("Hybrid Recommender ready!")

In [ ]:
# Test hybrid
pred = df_clean[df_clean['englishTitle'].str.contains('Predator', na=False)]
if len(pred) > 0:
    pred_id = pred['id'].iloc[0]
    print(f"User liked: {pred['englishTitle'].iloc[0]}")
    recs = hybrid_recommender.recommend(movie_id=pred_id, liked_genres=['Action'], n=5)
    print("\nHybrid recommendations:")
    print(recs.to_string(index=False))

print("\nGenre-based recommendations (Drama, Romance):")
recs = hybrid_recommender.recommend(liked_genres=['Drama', 'Romance'], n=5)
print(recs.to_string(index=False))

## 7. Evaluation

In [ ]:
import random

def evaluate_recommender(recommender, df, n_recommendations=10, n_users=15):
    """Evaluate based on genre overlap"""
    precisions = []
    recalls = []
    
    sample_indices = random.sample(range(len(df)), min(n_users, len(df)))
    
    for idx in sample_indices:
        movie = df.iloc[idx]
        genres = movie['genre_list']
        
        if not genres or not isinstance(genres, list):
            continue
        
        test_genres = set(genres)
        
        # Get recommendations
        if hasattr(recommender, 'get_similar_movies'):
            recs = recommender.get_similar_movies(movie['id'], n=n_recommendations)
        else:
            recs = recommender.recommend(movie_id=movie['id'], n=n_recommendations)
        
        if recs is None or len(recs) == 0:
            continue
        
        # Check genre overlap
        overlaps = []
        for _, row in recs.iterrows():
            rec_genres = row['genre_list']
            if isinstance(rec_genres, list):
                overlaps.append(len(test_genres & set(rec_genres)) > 0)
            else:
                overlaps.append(False)
        
        precision = sum(overlaps) / len(overlaps) if overlaps else 0
        recall = 1 if any(overlaps) else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    return {
        'precision': np.mean(precisions) if precisions else 0,
        'recall': np.mean(recalls) if recalls else 0
    }

print("Evaluating Content-Based...")
content_metrics = evaluate_recommender(content_recommender, df_clean)
print(f"Content-Based: {content_metrics}")

print("\nEvaluating Hybrid...")
hybrid_metrics = evaluate_recommender(hybrid_recommender, df_clean)
print(f"Hybrid: {hybrid_metrics}")

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Precision', 'Recall']
content_scores = [content_metrics['precision'], content_metrics['recall']]
hybrid_scores = [hybrid_metrics['precision'], hybrid_metrics['recall']]

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, content_scores, width, label='Content-Based', color='steelblue')
ax.bar(x + width/2, hybrid_scores, width, label='Hybrid', color='coral')
ax.set_ylabel('Score')
ax.set_title('Model Evaluation')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("="*50)
print("RECOMMENDER SYSTEM SUMMARY")
print("="*50)
print(f"""
Dataset:
  • Movies: {len(df_clean)}
  • Genres: {len(all_unique_genres)}
  • Year range: {df_clean['year_numeric'].min():.0f}-{df_clean['year_numeric'].max():.0f}

Features:
  • Genre one-hot: {len(genre_cols)}
  • Numeric: {len(numeric_cols)}
  • Total: {len(feature_cols)}

Models:
  1. Content-Based (cosine similarity)
  2. Popularity-Based (views, likes, ratings)
  3. Hybrid (content + popularity)

Results:
  • Content: Precision={content_metrics['precision']:.2f}, Recall={content_metrics['recall']:.2f}
  • Hybrid: Precision={hybrid_metrics['precision']:.2f}, Recall={hybrid_metrics['recall']:.2f}
""")